In [30]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from pathlib import Path
import json

In [24]:
def min_max_normalization(data, zero_impute=True):
    contain_na = False
    for omics_data in data:
        if np.isnan(omics_data).any():
            contain_na = True
    # min-max normalization
    if contain_na:
        new_data = []
        omics_min = []
        omics_max = []
        for omics_data in data:
            omics_data = pd.DataFrame(omics_data)
            min_ = omics_data.min()
            max_ = omics_data.max()
            omics_data = (omics_data - min_) / (max_ - min_) + 0.1
            omics_data = np.array(omics_data)

            if zero_impute:
                omics_data[np.isnan(omics_data)] = 0.1

            new_data.append(omics_data)
            omics_min.append(np.array(min_))
            omics_max.append(np.array(max_))
    else:
        new_data = []
        omics_min = []
        omics_max = []
        for omics_data in data:
            min_ = omics_data.min()
            max_ = omics_data.max()
            omics_data = (omics_data - min_) / (max_ - min_) + 0.1

            new_data.append(np.array(omics_data))
            omics_min.append(np.array([min_]))
            omics_max.append(np.array([max_]))

    return new_data, omics_min, omics_max

def KNN_impute(matrix):
    data = np.hstack((matrix))
    imputer = KNNImputer()
    KNN_impute_data = imputer.fit_transform(data)
    if data.shape != KNN_impute_data.shape:
        print("Skip KNN preimpute because of the all-zero features.")

        return matrix

    split_list = [0]
    for each in matrix:
        split_list.append(each.shape[1] + split_list[-1])
    matrix = np.split(KNN_impute_data, split_list[1:-1], axis=1)

    return matrix

def preprocess(matrix, normalization=True, pre_impute=False):
        # min-max normizaltion
    if normalization:
        matrix, omics_min, omics_max = min_max_normalization(data=matrix, zero_impute=False)
        matrix = np.array(matrix)

    # KNN preimpute
    if pre_impute:
        matrix = KNN_impute(matrix)
        matrix = np.array(matrix)

    return matrix


In [46]:
def load_edges_and_weights(matrix):
    a, b, c = np.nonzero(~np.isnan(matrix))
    edges = np.vstack((b, c, a)).T
    nums_type = 1 + edges.max(0)
    weights = matrix[a, b, c].reshape(-1, 1)
    return edges, weights, nums_type


def save_edges_and_weights(save_dir, edges, weights, nums_type, train_size):
    train_data, test_data, train_weight, test_weight = train_test_split(edges, weights, train_size=train_size)

    np.savez(save_dir / 'train_data.npz', train_data=train_data, train_weight=train_weight,
             nums_type=nums_type)
    np.savez(save_dir / 'test_data.npz', test_data=test_data, test_weight=test_weight,
             nums_type=nums_type)

def save_metadata(save_dir, edges, train_size, nums_type, normalization):
    metadata = dict(
        train_size=train_size,
        normalization=normalization,
        num_edges=edges.shape[0],
        num_nodes=sum(nums_type),
        num_cells=nums_type[0],
        num_genes=nums_type[1],
        num_omics=nums_type[2],
    )
    np.savez(save_dir / 'metadata', **metadata)

def create_hypergraph(matrix, save_dir, normalization=True, pre_impute=False, train_size=0.8):
    save_dir = Path(save_dir)
    matrix = preprocess(matrix, normalization=normalization, pre_impute=pre_impute)
    edges, weights, nums_type = load_edges_and_weights(matrix)
    save_edges_and_weights(save_dir, edges, weights, nums_type, train_size)
    save_metadata(save_dir, edges, train_size, nums_type, normalization)
    print('ok')


## sc-GEM

In [26]:
from examples.case_studies.sc_GEM_analysis import load_data

expression_data, methylation_data, labels = load_data()
data = np.array([expression_data, methylation_data])

In [47]:
create_hypergraph(data, './data/sc_GEM')

ok


In [58]:
x = np.load('./data/sc_GEM/metadata.npz', allow_pickle=True)

In [59]:
x

NpzFile './data/sc_GEM/metadata.npz' with keys: train_size, normalization, num_edges, num_nodes, num_cells...

In [61]:
dict(x)

{'train_size': array(0.8),
 'normalization': array(True),
 'num_edges': array(13440),
 'num_nodes': array(285),
 'num_cells': array(224),
 'num_genes': array(59),
 'num_omics': array(2)}